# LIBERO shared-autonomy experiments — success-rate analysis

Analyses every run written by [`experiment.py`](experiment.py) into
`outputs/pi05_libero_experiments/`. Each run directory holds `config.yaml`
(the resolved configuration, including the corruption / adapter matrices that
were in force), `trials.jsonl` (one record per completed trial) and, per
trial, a `.npz` of per-step arrays and an `.mp4`.

All runs are loaded by default; set `RUN_DIRS` by hand to look at a subset.

> **Read the caveats at the bottom before quoting any number from here.**
> Each condition has ~10 trials from a single operator — far too few to
> separate the conditions statistically.

Note on one field: in runs recorded before 2026-08-28 the `user_reads` value in
`trials.jsonl` is a session-cumulative counter rather than a per-trial one.
This notebook derives per-trial reads from the `.npz` step arrays instead, so
it is correct for old and new runs alike.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

sys.path.insert(0, str(Path.cwd()))  # notebooks/ on the path when the kernel starts here
sys.path.insert(0, str(Path.cwd() / "examples/pi05/libero_shared_autonomy/notebooks"))  # or at the repo root
try:
    import analyze
except ImportError:
    found = next(
        (
            p
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "examples/pi05/libero_shared_autonomy/notebooks/analyze.py").exists()
        ),
        None,
    )
    if found is None:
        raise ImportError("analyze.py not found under cwd or any parent directory") from None
    sys.path.insert(0, str(found / "examples/pi05/libero_shared_autonomy/notebooks"))
    import analyze

CANDIDATES = [Path("../../../../outputs/pi05_libero_experiments"), Path("outputs/pi05_libero_experiments")]
RUNS_ROOT = next((p for p in CANDIDATES if p.exists()), None)
if RUNS_ROOT is None:
    raise FileNotFoundError(f"no experiment output directory found; tried {[str(p) for p in CANDIDATES]}")
RUN_DIRS = analyze.find_runs(RUNS_ROOT)
print(f"{len(RUN_DIRS)} runs with data:")
for p in RUN_DIRS:
    print(" ", p.name)

## Load the runs

`trials.jsonl` holds one JSON object per *completed* trial; a skipped trial
leaves no record, so `trial` indices can have gaps. Each run gets a short
label built from what actually varied: the control mode, whether the operator's
command was corrupted by a matrix `M`, and whether the flow reversal used an
adapter `F`.

In [ ]:
trials, configs = analyze.load_runs(RUN_DIRS)
LABEL = {run: analyze.label_for(c) for run, c in configs.items()}
ORDER = list(LABEL)
print(f"{len(trials)} trials over {trials['run'].nunique()} runs")
trials[["run", "label", "trial", "task_id", "success", "steps", "duration_s"]].head()

## What was actually run

Anything that differs between runs other than the intended factor is a
confound, so list the settings that are not identical across all of them.

In [ ]:
keys = [
    "mode",
    "tau",
    "n_reverse_steps",
    "suite",
    "prompt",
    "n_trials",
    "seed",
    "task_order",
    "input_noise",
    "max_steps",
    "corruption",
    "reversal_adapter",
    "policy_path",
    "n_action_steps",
]
print("Settings that differ between runs (everything else is identical):")
analyze.differing_settings(configs, keys)

## The matrices that were in force

`M` (3x3) multiplies the operator's x/y/z command at every control step;
`F` (7x7) multiplies the policy's velocity field during the *reverse*
integration of `shared_reverse_flow_steering`. Both are recorded into the run's
`config.yaml`, so the analysis does not depend on those YAML files still
holding the same values.

In [ ]:
AXES = analyze.AXES
for run in ORDER:
    config = configs[run]
    print(f"=== {LABEL[run]}  ({run})")
    M, F = config.get("corruption_matrix"), config.get("flow_adapter_matrix")
    if M is None:
        print("  M: none, the operator's command is passed through untouched")
    else:
        M = np.array(M)
        print(f"  M: {analyze.describe_rotation(M)}")
        display(pd.DataFrame(M, index=AXES[:3], columns=AXES[:3]))
    if F is None:
        print("  F: none, the flow reversal uses the policy's own velocity field")
    else:
        F = np.array(F)
        zero_rows = [AXES[i] for i in range(7) if np.allclose(F[i], 0)]
        print(f"  F: translation block is {analyze.describe_rotation(F[:3, :3])}")
        if zero_rows:
            print(
                f"     rows that are entirely zero: {', '.join(zero_rows)} -> the reversal does not move those dims"
            )
        display(pd.DataFrame(F, index=AXES, columns=AXES))
    print()

## Success rate per run

The point estimate plus a **Wilson 95% interval**, which behaves sensibly for
small samples and proportions near 0 or 1 (unlike the normal approximation).

In [ ]:
summary = analyze.success_table(trials, configs)
summary.style.format({"success_rate": "{:.0%}", "ci95_low": "{:.0%}", "ci95_high": "{:.0%}"})

In [ ]:
fig, ax = plt.subplots(figsize=(1.9 * len(summary) + 2.5, 3.8))
x = np.arange(len(summary))
rate = summary["success_rate"].to_numpy()
err = np.vstack([rate - summary["ci95_low"], summary["ci95_high"] - rate])
colours = ["#4c8fbd" if m == "shared_flow_control" else "#c8814b" for m in summary["mode"]]
bars = ax.bar(
    x,
    rate,
    width=0.6,
    color=colours,
    hatch=["//" if c else "" for c in summary["corrupted"]],
    edgecolor="white",
    linewidth=0,
)
for bar in bars:
    bar.set_edgecolor("#ffffff")
ax.errorbar(x, rate, yerr=err, fmt="none", ecolor="#333", capsize=6, lw=1.4)
for xi, (r, n, k) in enumerate(zip(rate, summary["trials"], summary["successes"], strict=True)):
    ax.text(xi, r + 0.04, f"{r:.0%}\n({k}/{n})", ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(summary["label"], fontsize=9)
ax.set_ylim(0, 1.2)
ax.set_ylabel("success rate")
ax.set_title("Success rate with Wilson 95% CI  (hatched = operator command corrupted by M)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Collapsed by factor

The runs form a mode x corruption grid. **Caveat:** if one cell also carries a
flow adapter `F`, that cell is not a clean mode x corruption comparison — the
adapter is confounded with it. The `adapted` column above tells you which.

In [ ]:
grid = trials.pivot_table(index="mode", columns="corrupted", values="success", aggfunc=["mean", "count"])
grid.columns = [f"{stat} | {'M applied' if corrupted else 'clean'}" for stat, corrupted in grid.columns]
display(grid.style.format(lambda v: f"{v:.0%}" if isinstance(v, float) else f"{v:.0f}"))

print("Marginal rates (ignore these if the cells are confounded):")
print(trials.groupby("mode")["success"].agg(["mean", "count"]).rename(columns={"mean": "success_rate"}))
print()
print(trials.groupby("corrupted")["success"].agg(["mean", "count"]).rename(columns={"mean": "success_rate"}))

## Paired comparisons

Every run used the same suite, seed and `task_order`, so the trial schedule is
identical: trial *i* is the **same task** in every run. That makes this a
paired design, which is far more sensitive than comparing independent
proportions. Trials missing from a run (a skip) drop out of that pairing.

For paired binary outcomes the right test is **McNemar's**: only trials where
two conditions disagree carry information, so the exact binomial test is
applied to the discordant pairs.

In [ ]:
wide = trials.pivot_table(index="trial", columns="run", values="success", aggfunc="first")[ORDER]
tasks = trials.drop_duplicates("trial").set_index("trial")["task_id"]
wide.columns = [LABEL[r] for r in wide.columns]
display(
    wide.assign(task_id=tasks.reindex(wide.index))
    .set_index("task_id", append=True)
    .style.format(lambda v: "" if pd.isna(v) else ("success" if v else "fail"))
)
pairwise = analyze.paired_comparisons(trials)
pairwise["A"], pairwise["B"] = pairwise["A"].map(LABEL), pairwise["B"].map(LABEL)
pairwise.style.format({"A_rate": "{:.0%}", "B_rate": "{:.0%}", "mcnemar_p": "{:.3f}"})

In [ ]:
print("Smallest achievable two-sided p for a given number of discordant pairs:")
for d in range(1, 9):
    print(
        f"  {d} discordant -> p >= {stats.binomtest(0, d, 0.5).pvalue:.3f}"
        f"{'   (can never reach 0.05)' if stats.binomtest(0, d, 0.5).pvalue > 0.05 else ''}"
    )
print("\nSo with <= 5 discordant trials no result here can be significant, whatever the direction.")

In [ ]:
outcomes = wide.to_numpy(dtype=float)
fig, ax = plt.subplots(figsize=(1.0 * len(wide) + 3.5, 0.6 * len(wide.columns) + 2.2))
ax.imshow(np.ma.masked_invalid(outcomes.T), cmap=plt.cm.RdYlGn, vmin=0, vmax=1, aspect="auto")
for j in range(outcomes.shape[1]):
    for i in range(outcomes.shape[0]):
        v = outcomes[i, j]
        ax.text(
            i,
            j,
            "skipped" if np.isnan(v) else ("win" if v else "fail"),
            ha="center",
            va="center",
            fontsize=8,
            color="#222",
        )
ax.set_xticks(range(len(wide)))
ax.set_xticklabels([f"t{t}\ntask {int(tasks[t])}" for t in wide.index], fontsize=8)
ax.set_yticks(range(len(wide.columns)))
ax.set_yticklabels(wide.columns, fontsize=9)
ax.set_title("Per-trial outcomes (same task in each column across all runs)")
plt.tight_layout()
plt.show()

## Per-task view

Tasks repeat within a run (the schedule samples with replacement), so this
also shows how consistent a condition is on the same task.

In [ ]:
per_task = trials.pivot_table(
    index=["task_id", "task_description"], columns="label", values="success", aggfunc="mean"
).sort_index()
per_task["n_conditions_solved"] = (per_task > 0).sum(axis=1)
per_task.style.format(lambda v: f"{v:.0%}" if isinstance(v, float) and v <= 1 else f"{v:.0f}")

## How long trials took

`steps` is the control-step count (20 Hz). A trial that hits the episode limit
is a timeout, so successes and failures must be read separately — mixing them
makes the mean meaningless.

In [ ]:
limit = trials["steps"].max()
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(7.5, 0.75 * len(ORDER) + 2))
for i, run in enumerate(ORDER):
    group = trials[trials["run"] == run]
    for flag, colour, name in ((True, "#3f8f4f", "success"), (False, "#b1483c", "timeout/fail")):
        subset = group.loc[group["success"] == flag, "steps"]
        ax.scatter(
            subset,
            np.full(len(subset), i) + rng.normal(0, 0.05, len(subset)),
            color=colour,
            s=70,
            alpha=0.85,
            label=name if i == 0 else None,
            zorder=3,
        )
ax.axvline(limit, ls="--", lw=1, color="#888")
ax.text(limit, -0.45, f" episode limit ({int(limit)})", fontsize=8, color="#666", va="bottom")
ax.set_yticks(range(len(ORDER)))
ax.set_yticklabels([LABEL[r] for r in ORDER], fontsize=9)
ax.set_xlabel("control steps")
ax.set_title("Steps per trial")
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

trials.groupby("label", sort=False).apply(
    lambda g: pd.Series(
        {
            "median steps (success)": g.loc[g["success"], "steps"].median(),
            "median seconds (success)": g.loc[g["success"], "duration_s"].median(),
            "timeouts": int((g["steps"] >= limit).sum()),
        }
    ),
    include_groups=False,
)

## How much the operator actually steered

Each mode consumes the teleop input differently, so raw sample counts are
**not** comparable across modes: `shared_flow_control` reads the input once per
guided denoising step (`tau` per chunk) while `shared_reverse_flow_steering`
reads it once per chunk. What *is* comparable is what the `.npz` records per
control step: the operator's command before any corruption
(`user_translation_raw`), i.e. how much of the trial they spent pushing.

In [ ]:
activity = analyze.input_activity(trials)
activity.groupby("label", sort=False)[
    ["commanding_frac", "mean_speed_when_active", "mean_corruption_shift", "gripper_closed_frac"]
].mean().round(3)

In [ ]:
# `reads_this_trial` comes from the step arrays; the mode-specific columns only
# exist for the mode that produces them, hence the NaNs.
metrics = [
    "reads_this_trial",
    "n_steps",
    "guided_denoising_steps",
    "steered_chunks",
    "reconstruction_error_mean",
]
present = [m for m in metrics if m in activity.columns]
steering = activity.groupby("label", sort=False)[present].mean()
steering["reads_per_step"] = steering["reads_this_trial"] / steering["n_steps"]
print("Mean per trial (NaN = the metric does not apply to that mode):")
steering.round(2)

In [ ]:
fig, axes = plt.subplots(1, len(ORDER), figsize=(3.4 * len(ORDER), 3.1), sharey=True, squeeze=False)
for ax, run in zip(axes[0], ORDER, strict=True):
    group = activity[activity["run"] == run]
    ok = group["success"].astype(bool)
    ax.scatter(
        group.loc[ok, "commanding_frac"], group.loc[ok, "steps"], color="#3f8f4f", s=70, label="success"
    )
    ax.scatter(
        group.loc[~ok, "commanding_frac"], group.loc[~ok, "steps"], color="#b1483c", s=70, label="fail"
    )
    ax.set_title(LABEL[run], fontsize=10)
    ax.set_xlabel("fraction of steps pushing")
    ax.set_xlim(-0.03, 1.03)
    ax.spines[["top", "right"]].set_visible(False)
axes[0][0].set_ylabel("steps")
axes[0][0].legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

## Findings and caveats

Run the cell below for the headline numbers of whatever runs are loaded.

**Caveats:**

1. **The samples are far too small.** ~10 trials per condition. The Wilson
   intervals span roughly 20-80% and overlap heavily, and the paired tests rest
   on a handful of discordant trials — see the "smallest achievable p" cell:
   with 5 or fewer discordant trials, no result can reach p < 0.05 in either
   direction.
2. **One operator, unblinded, runs back to back.** The same person drove every
   condition knowing which was which, on the same task schedule each time.
   Practice accumulates across runs and is inseparable from condition here;
   in chronological order it works *against* later runs looking worse and *for*
   them looking better.
3. **A confounded cell.** If a run applies both a corruption `M` and an adapter
   `F`, its result cannot be attributed to either alone — check the `adapted`
   column of the summary before reading the mode x corruption grid.
4. **A skipped trial breaks one pair**, so unpaired and paired rates differ
   slightly for that run.
5. **Success is the environment's own flag** at episode end. It says nothing
   about *how* the task was accomplished — in particular, not how much of the
   work was the operator's versus the policy's.
6. **Timeouts dominate the failures**, so `steps` mostly measures the episode
   limit for failures; only compare durations among successes.

A powered version of this comparison needs a pre-registered trial count (order
of 100+ per condition), condition order counterbalanced, one factor varied at a
time, and ideally several operators.

In [ ]:
print("Success rate per run (chronological):\n")
for run in ORDER:
    row = summary.loc[run]
    flags = []
    if row["corrupted"]:
        flags.append("command corrupted by M")
    if row["adapted"]:
        flags.append("reversal adapted by F")
    print(
        f"  {row['label']:14s} {row['successes']}/{row['trials']} = {row['success_rate']:.0%} "
        f"(95% CI {row['ci95_low']:.0%}-{row['ci95_high']:.0%})"
        + (f"   [{'; '.join(flags)}]" if flags else "")
    )

best, worst = summary["success_rate"].idxmax(), summary["success_rate"].idxmin()
print(
    f"\nHighest: {summary.loc[best, 'label']} ({summary.loc[best, 'success_rate']:.0%}); "
    f"lowest: {summary.loc[worst, 'label']} ({summary.loc[worst, 'success_rate']:.0%})."
)

significant = pairwise[pairwise["mcnemar_p"] < 0.05]
print(f"\nPairwise comparisons reaching p < 0.05: {len(significant)} of {len(pairwise)}.")
scored = pairwise.dropna(subset=["mcnemar_p"])
if scored.empty:
    print("No pairwise comparison had a discordant trial, so no p-value could be computed.")
else:
    best_row = scored.loc[scored["mcnemar_p"].idxmin()]
    print(
        f"Most suggestive contrast: {best_row['A']} vs {best_row['B']} "
        f"(p = {best_row['mcnemar_p']:.3f}) — still not significant."
    )